# CHO rate-reconciled fed-batch: editable native PharmaPy workflow

This notebook contains the tasks performed by the example scripts directly: configuration, native construction, forward simulation, reporting, plotting, inference, and regression checks. It does **not** import or execute `run_*.py` helpers. Reusable modeling, estimation, and experiment-design algorithms remain in `PharmaPy`.

Use the configured **Python 3.11** environment described in the parent README as the notebook kernel. The complete design study performs repeated fits and refinement checks and can take **tens of minutes**. Outputs are held in memory by default; the last step can export to a separate destination.

For a new example, copy this notebook and compatible `inputs/` files, change the paths and configuration steps, and provide scientifically justified model/property definitions. A new organism is not inferred from its name. These templates assume a declared **constant observation density**. Synthetic observations demonstrate methodology, not independent experimental validation.

The standalone scripts remain available for command-line reproduction. Their original thresholds, seeds, and calculations are retained here; comparisons against the distributed outputs are automatically skipped for changed configurations.

**Forward simulation only:** execute Steps 1–6 and stop. Continue to inference only after reviewing the separate estimation/design settings; those studies have their own operating overrides and sampling times.


## Step 1 — Imports and input locations

Set example_dir to your copied example folder when adapting this notebook. The repository must be importable; the automatic search works when opened inside this checkout. No example files are changed.

In [ ]:
# STEP 1: Imports and input locations
# Set example_dir to your copied example folder when adapting this notebook.
# The repository must be importable; the automatic search works when opened inside this
# checkout. No example files are changed.

from pathlib import Path
from copy import deepcopy
import csv
import hashlib
import json
import sys
import numpy as np
import matplotlib.pyplot as plt

# If the notebook is outside this checkout, set repo_root explicitly instead.
repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / "PharmaPy").is_dir()), None)
if repo_root is None:
    raise ValueError("Set repo_root to your PharmaPy checkout before running this cell")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from PharmaPy.Bioreactors import (
    build_bioreactor, BioreactorEstimationProblem, ObservationSeries,
    estimate_bioreactor_parameters,
)
from PharmaPy.Bioreactors.experiments import native_experiment_model, run_design_study

example_dir = repo_root / "examples/bioreactors/fed_batch_cho"
# For a copied example, replace the line above with the absolute path.
reference_dir = repo_root / "examples/bioreactors/fed_batch_cho"
input_dir = example_dir / "inputs"
thermo_path = input_dir / "thermo.json"
WRITE_ARTIFACTS = False
EXPORT_DIR = example_dir / "custom_results"
COMPARE_CANONICAL = True


## Step 2 — Load and edit model and study configuration

Edit the dictionaries here or the JSON files before running. Keep mechanism identifiers, species/property mappings, recipe names, and duplicated initial volumes consistent. Initial pathway amounts are mol and biomass kgDW; culture amounts are mmol, cells million cells, product g. Update study times, parameter paths, noise, and candidate settings when changing biology. Example overrides below are commented out.

In [ ]:
# STEP 2: Load and edit model and study configuration
# Edit the dictionaries here or the JSON files before running. Keep
# mechanism identifiers, species/property mappings, recipe names, and duplicated initial
# volumes consistent. Initial pathway amounts are mol and biomass kgDW; culture amounts
# are mmol, cells million cells, product g. Update study times, parameter paths, noise,
# and candidate settings when changing biology. Example overrides below are commented
# out.

case = json.loads((input_dir / "case.json").read_text())
config = json.loads((input_dir / "mechanism.json").read_text())
design_settings = json.loads((input_dir / "design.json").read_text())

# EXAMPLES ONLY: uncomment and choose values appropriate for your model.
# case["operation"]["runtime"]["value"] = 12.0
# case["biology"]["conditions"]["ph"] = 7.0  # culture provider only
# config["state"]["species_mol"]["glucose"] = 0.020  # pathway model only
# design_settings["criterion"] = "A"  # D, A, or E
# Event times use numerics.step.unit. Fixed-step observations must lie on its mesh.
# A new volume also requires matching initial_mass_kg, mechanism state volume,
# and physically consistent species/cell inventories.
# For new settings, review acceptance limits rather than weakening failed checks.
print(json.dumps(case, indent=2))
print("Design parameters:", design_settings["parameters"])
print("Candidate experiments:", design_settings["candidates"])


## Step 3 — Construct and run the native reactor

Run this cell to build the LiquidPhase, biological mechanism, and native reactor and execute the declared schedule. histories retains every segment; unit.result alone is only the latest segment. Use a fresh assembly after configuration changes.

In [ ]:
# STEP 3: Construct and run the native reactor
# Run this cell to build the LiquidPhase, biological mechanism, and native
# reactor and execute the declared schedule. histories retains every segment;
# unit.result alone is only the latest segment. Use a fresh assembly after configuration
# changes.

assembly = build_bioreactor(case, config, thermo_path)
unit = assembly.unit
phase = assembly.phase
species = tuple(assembly.mechanism.definition.extracellular_species)
histories = assembly.solve(verbose=False)


## Step 4 — Extract trajectories with explicit units

Run to convert recorded inventories using the configured constant density at each time. The display table keeps pre-event boundary rows; histories retains pre/post states for observation selection. Change output quantities here when adapting the model. Do not use these conversions for variable-density physics without supplying a suitable volume history.

In [ ]:
# STEP 4: Extract trajectories with explicit units
# Run to convert recorded inventories using the configured constant density
# at each time. The display table keeps pre-event boundary rows; histories retains
# pre/post states for observation selection. Change output quantities here when adapting
# the model. Do not use these conversions for variable-density physics without supplying
# a suitable volume history.

columns = ["time_day", *species, "liquid_volume_l", "viable_cells_million",
           "dead_cells_million", "product_g", "ivcd_million_cell_day_per_ml"]
rows = []
for segment_index, result in enumerate(histories):
    mass_history = np.asarray(result.mass_j_liquid0)
    start_index = 0 if segment_index == 0 else 1
    for i in range(start_index, len(result.time)):
        mass = mass_history[i]
        volume_l = mass.sum() / case["operation"]["carrier_density_kg_l"]
        row = {"time_day": result.time[i] / 86400.0}
        for name in species:
            j = list(phase.name_species).index(name)
            row[name] = mass[j] / phase.mw[j] * 1e6 / volume_l
        row.update({
            "liquid_volume_l": volume_l,
            "viable_cells_million": result.viable_cells_million_liquid0[i],
            "dead_cells_million": result.dead_cells_million_liquid0[i],
            "product_g": result.product_g_liquid0[i],
            "ivcd_million_cell_day_per_ml": result.ivcd_million_cell_day_per_ml_liquid0[i],
        })
        rows.append(row)

print("Recorded rows:", len(rows))
print("Final row:", rows[-1])


## Step 5 — Check, summarize, and validate forward results

Inspect finiteness, nonnegative inventories, and the optional original endpoint regression. These checks are not complete elemental conservation or experimental validation. NOT_CHECKED means no original-example comparison was made.

***Note***: The canonical comparison is only applicable to the unchanged distributed example. Finiteness and inventory checks remain applicable to modified or new simulations.

In [ ]:
# STEP 5: Check, summarize, and validate forward results
# Inspect finiteness, nonnegative inventories, and the optional original
# endpoint regression. These checks are not complete elemental conservation or
# experimental validation. NOT_CHECKED means no original-example comparison was made.

# Compare with distributed evidence only when every original input is unchanged.
configured_documents = {"case.json": case, "mechanism.json": config, "design.json": design_settings}
REFERENCE_INPUT_SHA256 = {
    "case.json": "124b1e497fc8c94ecb627b5deb08aa39832a0bbcd637578190b7c0e8ba33f7d9",
    "mechanism.json": "a613fea5a62c4900a766a8c7b9f010b2203871c7b9d0f0d3b704859897d5bf44",
    "design.json": "ca000932a1cdb063a8c159a4147d2795aa054acc4f1760ac201bc88082f09029",
    "thermo.json": "5a7073e56099c9a76fb4bfdf45a4c5a8bf603b392c5fa7ec99898d256a52fc24"
}
reference_inputs_match = all(
    hashlib.sha256(json.dumps(document, sort_keys=True).encode()).hexdigest()
    == REFERENCE_INPUT_SHA256[name]
    for name, document in configured_documents.items()
) and hashlib.sha256(thermo_path.read_bytes()).hexdigest() == REFERENCE_INPUT_SHA256["thermo.json"]
check_regression = COMPARE_CANONICAL and reference_inputs_match
print("Original-example regression:", "ENABLED" if check_regression else "NOT_CHECKED")

final_vcd = rows[-1]["viable_cells_million"] / (rows[-1]["liquid_volume_l"] * 1000.0)
baseline = (json.loads((reference_dir / 'inputs/regression_baseline.json').read_text())
            if check_regression else None)
reference_vcd = baseline['endpoints']['final_vcd_million_ml'] if baseline else None
reference_product = (baseline['endpoints']['final_product_g'] if baseline
                     else None)
vcd_error = abs(final_vcd - reference_vcd) / reference_vcd if baseline else None
product_error = (abs(rows[-1]["product_g"] - reference_product) / reference_product
                 if baseline else None)
status = ('NOT_CHECKED' if baseline is None else 'PASS'
          if max(vcd_error, product_error) < baseline['relative_tolerance'] else 'FAIL')
summary = {"status": status, "native_unit": type(unit).__name__,
           "final_vcd_million_ml": final_vcd,
           "final_product_g": rows[-1]["product_g"],
           "relative_final_vcd_error": vcd_error,
           "relative_final_product_error": product_error,
           "comparison_kind": "generated_model_regression_not_experimental_validation"}
numeric = np.array([[row[name] for name in columns[1:]] for row in rows])

balance_audit = {"finite": bool(np.isfinite(numeric).all()), "minimum_reported_value": float(numeric.min()), "passed": bool(np.isfinite(numeric).all() and numeric.min() >= -1e-10)}
assert balance_audit["finite"]
assert balance_audit.get("nonnegative_inventory_constraint", balance_audit.get("passed"))
if check_regression:
    assert summary["status"] == "PASS", "forward endpoint regression failed"
simulation = {"summary": summary, "rows": rows}
print(json.dumps(summary, indent=2))
print(balance_audit)


## Step 6 — Plot forward outputs

Run to display the native simulation results. Adjust species selection and labels for your case; underlying balances are unchanged. The two examples use different output bases; inspect axis labels before comparison.

In [ ]:
# STEP 6: Plot forward outputs
# Run to display the native simulation results. Adjust species selection and
# labels for your case; underlying balances are unchanged. The two examples use
# different output bases; inspect axis labels before comparison.

figure, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True, constrained_layout=True)
time = [row["time_day"] for row in rows]
# Edit this display selection for a new culture; all species remain in rows.
plot_species = [name for name in ("GLC", "GLN", "LAC", "NH3") if name in species]
if not plot_species:
    plot_species = list(species[:6])
for name in plot_species:
    axes[0].plot(time, [row[name] for row in rows], label=name)
axes[1].plot(time, [row["viable_cells_million"] /
                     (row["liquid_volume_l"] * 1000.0) for row in rows], label="VCD")
axes[2].plot(time, [row["product_g"] for row in rows], label="product mass")
axes[0].set(ylabel="Species (mmol/L)", title="Native rate-reconciled culture trajectory")
axes[1].set(ylabel="VCD (million cells/mL)")
axes[2].set(xlabel="Time (day)", ylabel="Product inventory (g)")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.show()


## Step 7 — Build fresh simulation callbacks for experiment design

Review design_settings from Step 2: parameter paths, candidate control paths, cost limits, observations/covariance, pre/post event timing, training/withheld times, seeds, and acceptance profile. Fixed-step sample times must lie on the mesh. This shared library adapter constructs a new native experiment per evaluation; no example runner is used.

In [ ]:
# STEP 7: Build fresh simulation callbacks for experiment design
# Review design_settings from Step 2: parameter paths, candidate control
# paths, cost limits, observations/covariance, pre/post event timing, training/withheld
# times, seeds, and acceptance profile. Fixed-step sample times must lie on the mesh.
# This shared library adapter constructs a new native experiment per evaluation; no
# example runner is used.

design_model = native_experiment_model(case, config, thermo_path, design_settings)
refined_design_model = native_experiment_model(
    case, config, thermo_path, design_settings,
    refinement=design_settings["refinement_factor"],
)
print("Criterion:", design_settings["criterion"])
print("Measurements:", design_settings["observations"])
print("Sample times (s):", design_settings["sample_times_s"])
print("Acceptance:", design_settings["acceptance"])


## Step 8 — Calibrate, select an experiment, and refit

Execute the full study; allow tens of minutes. The library coordinator fits baseline synthetic observations, computes finite-difference information, ranks candidates, replays selection, generates paired selected/control observations, refits, and checks numerical refinement. It does not request laboratory measurements. The true synthetic parameter values generate assessment data; selection uses fitted values.

In [ ]:
# STEP 8: Calibrate, select an experiment, and refit
# Execute the full study; allow tens of minutes. The library coordinator
# fits baseline synthetic observations, computes finite-difference information, ranks
# candidates, replays selection, generates paired selected/control observations, refits,
# and checks numerical refinement. It does not request laboratory measurements. The true
# synthetic parameter values generate assessment data; selection uses fitted values.

design = run_design_study(design_settings, design_model, refined_design_model)
assert design["status"] == "PASS", "experiment-design acceptance failed"
print(json.dumps({key: design[key] for key in (
    "status", "selected_id", "checks", "capability_status", "empirical_benefit_status",
    "predicted_sd_reduction_vs_matched_control", "mean_selected_prediction_mse",
    "mean_control_prediction_mse",
)}, indent=2))


**Interpretation:** capability acceptance and realized predictive benefit are separate. The CHO reference study reports `NOT_DEMONSTRATED` empirical benefit despite improved predicted precision. Do not substitute a favorable criterion, seed, or tolerance after seeing results. To fit real experiments, use `ObservationSeries` and `BioreactorEstimationProblem`; this combined coordinator specifically generates synthetic demonstration data.

## Step 9 — Compare the unchanged example with distributed evidence

This optional gate retains the original regression tolerances and checks. It only runs for unchanged inputs; custom cases receive NOT_CHECKED for reference agreement while study acceptance remains mandatory. Do not regenerate the original baseline just to make changed biology pass.

In [ ]:
# STEP 9: Compare the unchanged example with distributed evidence
# This optional gate retains the original regression tolerances and checks.
# It only runs for unchanged inputs; custom cases receive NOT_CHECKED for reference
# agreement while study acceptance remains mandatory. Do not regenerate the original
# baseline just to make changed biology pass.

if check_regression:
    canonical = reference_dir / "outputs"
    # Regression agreement: 0.01% relative, with a 1e-8 floor in each column's units.
    # This allows small optimizer/platform drift while remaining well below the 2% benchmark.
    REGRESSION_RTOL = 1e-4
    REGRESSION_ATOL = 1e-8
    # Error fractions use an absolute tolerance: 1e-7 = 0.00001 percentage points.
    ERROR_FRACTION_ATOL = 1e-7

    def assert_regression_close(actual, expected, name, *, rtol=REGRESSION_RTOL,
                                atol=REGRESSION_ATOL):
        assert np.all(np.isfinite(actual)), f"{name}: nonfinite generated values"
        assert np.all(np.isfinite(expected)), f"{name}: nonfinite canonical values"
        np.testing.assert_allclose(actual, expected, rtol=rtol, atol=atol,
                                   equal_nan=False, err_msg=name)

    generated_trajectory = simulation["rows"]
    canonical_trajectory = np.genfromtxt(canonical / "trajectories.csv", delimiter=",", names=True)
    for name in canonical_trajectory.dtype.names:
        # The declared time grid must agree exactly; it is not an optimized output.
        assert_regression_close(
            [row[name] for row in generated_trajectory], canonical_trajectory[name],
            f"CHO trajectory: {name}",
            rtol=0.0 if name == "time_day" else REGRESSION_RTOL,
            atol=0.0 if name == "time_day" else REGRESSION_ATOL,
        )
    trajectory_agrees = True
    canonical_summary = json.loads((canonical / "summary.json").read_text())
    canonical_design = json.loads((canonical / "experiment_design.json").read_text())
    for name in ("relative_final_vcd_error", "relative_final_product_error"):
        assert_regression_close(simulation["summary"][name], canonical_summary[name],
                                f"CHO summary: {name}", rtol=0.0,
                                atol=ERROR_FRACTION_ATOL)
        assert 0.0 <= simulation["summary"][name] < 0.02, f"CHO benchmark failed: {name}"
    assert design["selected_id"] == canonical_design["selected_id"], "CHO design selection changed"
    assert_regression_close(design["score_improvement"], canonical_design["score_improvement"],
                            "CHO design: score_improvement")
    assert simulation["summary"]["status"] == "PASS", "CHO simulation acceptance failed"
    assert design["status"] == "PASS", "CHO design acceptance failed"
    assert design["capability_status"] == "PASS", "CHO design capability failed"
    assert all(design["checks"][name] for name in design["required_checks"]), "CHO required design check failed"
    assert design["acceptance_profile"] == canonical_design["acceptance_profile"], "CHO acceptance profile changed"
    assert design["empirical_benefit_status"] == canonical_design["empirical_benefit_status"], "CHO empirical evidence changed"
    metrics_agree = True
    workflow_result = {"status": "PASS", "native_unit": simulation["summary"]["native_unit"], "simulation_status": simulation["summary"]["status"], "design_status": design["status"], "trajectory_agrees": trajectory_agrees, "metrics_agree": metrics_agree}
    workflow_result.update(acceptance_profile=design["acceptance_profile"], capability_status=design["capability_status"], empirical_benefit_status=design["empirical_benefit_status"])
    print('WORKFLOW_RESULT=' + json.dumps(workflow_result, sort_keys=True))

else:
    workflow_result = {"status": "PASS", "regression_status": "NOT_CHECKED",
                       "trajectory_agrees": None, "metrics_agree": None,
                       "native_unit": simulation["summary"]["native_unit"],
                       "simulation_status": simulation["summary"]["status"],
                       "design_status": design["status"],
                       "capability_status": design["capability_status"],
                       "empirical_benefit_status": design["empirical_benefit_status"]}
    print('WORKFLOW_RESULT=' + json.dumps(workflow_result, sort_keys=True))


## Step 10 — Optionally export this run

Default is no file writes. Set WRITE_ARTIFACTS=True in Step 1 and choose EXPORT_DIR to save a separate report and CSV; distributed outputs are protected. Segment mass files preserve both sides of material events. Copy your edited input dictionaries too so the run can be reproduced.

**Execute:** read the comments, edit only the indicated settings, then select the code cell below and press **Shift+Enter**. Comments beginning with `#` are instructions; Python executes the statements beneath them. Run steps in order. After changing inputs, restart the kernel and run again from Step 1.

In [ ]:
# STEP 10: Optionally export this run
# Default is no file writes. Set WRITE_ARTIFACTS=True in Step 1 and choose
# EXPORT_DIR to save a separate report and CSV; distributed outputs are protected.
# Segment mass files preserve both sides of material events. Copy your edited input
# dictionaries too so the run can be reproduced.

if WRITE_ARTIFACTS:
    output = EXPORT_DIR.resolve()
    protected = (reference_dir / "outputs").resolve()
    if output == protected or protected in output.parents:
        raise ValueError("Choose an export folder outside distributed reference outputs")
    output.mkdir(parents=True, exist_ok=True)
    with (output / "trajectories.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=simulation["rows"][0])
        writer.writeheader()
        writer.writerows(simulation["rows"])
    reports = {"summary": simulation["summary"], "balance_audit": balance_audit,
               "experiment_design": design, "workflow_result": workflow_result,
               "validation": {"status": simulation["summary"]["status"],
                              "independent_experimental_validation": False,
                              "criterion": "Original-model regression only when inputs match"}}
    if "estimation" in globals():
        reports["parameter_estimation"] = estimation
    for name, report in reports.items():
        (output / f"{name}.json").write_text(json.dumps(report, indent=2) + "\n")
    saved_inputs = output / "inputs"
    saved_inputs.mkdir(exist_ok=True)
    for name, document in configured_documents.items():
        (saved_inputs / name).write_text(json.dumps(document, indent=2) + "\n")
    (saved_inputs / "thermo.json").write_bytes(thermo_path.read_bytes())
    for index, segment in enumerate(histories):
        data = np.column_stack((segment.time, segment.mass_j_liquid0))
        np.savetxt(output / f"species_segment_{index:03d}.csv", data, delimiter=",",
                   header=",".join(["time_s"] + [f"{name}_kg" for name in assembly.phase.name_species]), comments="")
    figure.savefig(output / "trajectories.png", dpi=180)
    print("Saved run to", output)
else:
    print("Results remain in memory; no artifacts written.")
plt.close(figure)